Отлично, Лоро, сейчас сделаем тебе **идеальный OAuth flow как у продакшн-сервисов** 🔥
Чтобы я собрал всё **под тебя без багов**, скинь вот это:

---

# 📦 1. Твой текущий `.env` (без секретов)

PORT=5000
MONGO_URI=
FACEIT_KEY=
FACEIT_REDIRECT_URI=
FRONTEND_URL=
FACEIT_CLIENT_ID=
FACEIT_CLIENT_SECRET=
STEAM_API_KEY=
SESSION_SECRET=

# 🧠 2. Текущий `faceitOAuth.js` (ты уже почти скинул, но финальную версию)

const router = require("express").Router();
const axios = require("axios");
const crypto = require("crypto");
const User = require("../models/User");

function base64url(buffer) {
  return buffer
    .toString("base64")
    .replace(/\+/g, "-")
    .replace(/\//g, "_")
    .replace(/=+$/g, "");
}

function generateCodeVerifier() {
  return base64url(crypto.randomBytes(64));
}

function generateCodeChallenge(verifier) {
  return base64url(crypto.createHash("sha256").update(verifier).digest());
}

function buildAuthorizeUrl(state, codeChallenge) {
  const params = new URLSearchParams({
    client_id: process.env.FACEIT_CLIENT_ID,
    redirect_uri: process.env.FACEIT_REDIRECT_URI,
    response_type: "code",
    scope: "openid",
    state,
    code_challenge: codeChallenge,
    code_challenge_method: "S256",
  });

  return `https://accounts.faceit.com/oauth/authorize?${params.toString()}`;
}

// 1) redirect to FACEIT login
router.get("/login", (req, res) => {
  const state = crypto.randomBytes(16).toString("hex");
  const codeVerifier = generateCodeVerifier();
  const codeChallenge = generateCodeChallenge(codeVerifier);

  req.session.faceitState = state;
  req.session.faceitCodeVerifier = codeVerifier;

  res.redirect(buildAuthorizeUrl(state, codeChallenge));
});

// 2) callback after successful login
router.get("/callback", async (req, res) => {
  const { code, state, error } = req.query;

  if (error) {
    return res.status(400).json({ error });
  }

  if (!code) {
    return res.status(400).json({ error: "No authorization code received" });
  }

  if (!state || state !== req.session.faceitState) {
    return res.status(400).json({ error: "Invalid state" });
  }

  if (!req.session.faceitCodeVerifier) {
    return res.status(400).json({ error: "Missing PKCE code verifier" });
  }

  try {
    const basicAuth = Buffer.from(
      `${process.env.FACEIT_CLIENT_ID}:${process.env.FACEIT_CLIENT_SECRET}`,
    ).toString("base64");

    const tokenParams = new URLSearchParams({
      grant_type: "authorization_code",
      code,
      redirect_uri: process.env.FACEIT_REDIRECT_URI,
      code_verifier: req.session.faceitCodeVerifier,
    });

    // exchange code -> access token
    const tokenResponse = await axios.post(
      "https://api.faceit.com/auth/v1/oauth/token",
      tokenParams.toString(),
      {
        headers: {
          "Content-Type": "application/x-www-form-urlencoded",
          Authorization: `Basic ${basicAuth}`,
        },
      },
    );

    const { access_token } = tokenResponse.data;

    // get OpenID userinfo
    const userInfoResponse = await axios.get(
      "https://api.faceit.com/auth/v1/resources/userinfo",
      {
        headers: {
          Authorization: `Bearer ${access_token}`,
        },
      },
    );

    const userInfo = userInfoResponse.data;
    const faceitId = userInfo.sub;

    if (!faceitId) {
      return res.status(400).json({ error: "Faceit user id not found" });
    }

    // get full player data via FACEIT Data API
    const playerResponse = await axios.get(
      `https://open.faceit.com/data/v4/players/${faceitId}`,
      {
        headers: {
          Authorization: `Bearer ${process.env.FACEIT_KEY}`,
        },
      },
    );

    const player = playerResponse.data;
    const cs2 = player.games?.cs2 || player.games?.csgo || {};

    let user = await User.findOne({ faceitId });

    if (!user) {
      user = new User({
        faceitId,
        nickname: player.nickname || "FaceitUser",
        avatar: player.avatar || "",
        elo: cs2.faceit_elo || 0,
        level: cs2.skill_level || 0,
        country: (player.country || "unknown").toLowerCase(),
      });
    } else {
      user.nickname = player.nickname || user.nickname;
      user.avatar = player.avatar || user.avatar;
      user.elo = cs2.faceit_elo || user.elo || 0;
      user.level = cs2.skill_level || user.level || 0;
      user.country = (
        player.country ||
        user.country ||
        "unknown"
      ).toLowerCase();
    }

    await user.save();

    req.session.userId = user._id;

    delete req.session.faceitState;
    delete req.session.faceitCodeVerifier;

    req.session.save(() => {
      res.redirect(
        `${process.env.FRONTEND_URL}/auth/success?nickname=${encodeURIComponent(user.nickname)}`,
      );
    });
  } catch (err) {
    console.log("FACEIT OAuth error:", err.response?.data || err.message);
    return res.status(500).json({
      error: err.response?.data || err.message,
    });
  }
});

router.get("/me", async (req, res) => {
  if (!req.session.userId) {
    return res.status(401).json({ error: "Not authenticated" });
  }

  try {
    const user = await User.findById(req.session.userId);
    if (!user) {
      return res.status(404).json({ error: "User not found" });
    }
    return res.json(user);
  } catch (err) {
    return res.status(500).json({ error: err.message });
  }
});

router.post("/logout", (req, res) => {
  req.session.destroy(() => {
    res.json({ message: "Logged out" });
  });
});

module.exports = router;


# 🌐 3. Твой `AuthContext` (или где у тебя `fetchMe`)

import { createContext, useContext, useEffect, useState } from "react";

const AuthContext = createContext(null);

const API_URL = "https://matrix-8of6.onrender.com";

export function AuthProvider({ children }) {
  const [user, setUser] = useState(null);
  const [loading, setLoading] = useState(true);

  const fetchMe = async () => {
    try {
      const response = await fetch(`${API_URL}/auth/faceit/me`, {
        method: "GET",
        credentials: "include",
      });

      if (!response.ok) {
        setUser(null);
        return;
      }

      const data = await response.json();
      setUser(data);
    } catch (error) {
      console.error("Fetch me error:", error);
      setUser(null);
    } finally {
      setLoading(false);
    }
  };

  const logout = async () => {
    try {
      await fetch(`${API_URL}/auth/faceit/logout`, {
        method: "POST",
        credentials: "include",
      });
      setUser(null);
    } catch (error) {
      console.error("Logout error:", error);
    }
  };

  useEffect(() => {
    fetchMe();
  }, []);

  return (
    <AuthContext.Provider
      value={{
        user,
        setUser,
        loading,
        fetchMe,
        logout,
        apiUrl: API_URL,
      }}
    >
      {children}
    </AuthContext.Provider>
  );
}

export function useAuth() {
  return useContext(AuthContext);
}

# 🧭 4. Роуты фронта (`App.jsx` или router)
import { Route, Routes } from "react-router-dom";
import HomePage from "./pages/Home/HomePage";
import PlaygroundPage from "./pages/Playground/PlaygroundPage";
import NewsDetailPage from "./pages/NewsDetail/NewsDetailPage";
import LoginPage from "./pages/Login/LoginPage";
import RegisterPage from "./pages/Register/RegisterPage";
import AuthSuccess from "./pages/AuthSuccess";
import CounterStrafe from "./pages/PlayPages/CounterStrafe";
import PlayerProfile from "./components/sections/playerProfile/PlayerProfile";
import AdminPage from "./pages/Admin/AdminPage";
import TournamentsPage from "./pages/Tournaments/TournamentsPage";

const MainRoutes = () => {
  //! Сюда добавлять ссылки на страницы
  const PUBLIC_PAGES = [
    {
      link: "/",
      element: <HomePage />,
      id: 1,
    },
    {
      link: "/playground",
      element: <PlaygroundPage />,
      id: 2,
    },
    {
      link: "/news/details/:id",
      element: <NewsDetailPage />,
      id: 3,
    },
    {
      link: "/login",
      element: <LoginPage />,
      id: 4,
    },
    {
      link: "/register",
      element: <RegisterPage />,
      id: 5,
    },
    {
      link: "/auth/success",
      element: <AuthSuccess />,
      id: 6,
    },
    {
      link: "/playground/counter-strafe",
      element: <CounterStrafe />,
      id: 7,
    },
    {
      link: "/player/:id",
      element: <PlayerProfile />,
      id: 8,
    },
    {
      link: "/admin",
      element: <AdminPage />,
      id: 9,
    },
    {
      link: "/tournaments",
      element: <TournamentsPage />,
      id: 9,
    },
  ];

  return (
    <div>
      <Routes>
        {PUBLIC_PAGES.map((item) => (
          <Route path={item.link} element={item.element} key={item.id} />
        ))}
      </Routes>
    </div>
  );
};

export default MainRoutes;


# 🧱 5. Как ты вызываешь login (кнопка)
  const handleLogin = () => {
    const isMobile = /iPhone|iPad|iPod|Android/i.test(navigator.userAgent);

    if (isMobile) {
      window.location.href = `${apiUrl}/auth/faceit/login`;
    } else {
      window.open(
        `${apiUrl}/auth/faceit/login`,
        "faceitLogin",
        "width=600,height=700",
      );
    }
  };
  <button onClick={handleLogin} className="mx-loginbox__button">
              <span>
                Login <span className="fchide">with Faceit</span>
              </span>
              <img src={fcsmlogo} alt="e" className="logBtn_img" />
            </button>



